# Facial Expression Classification Using LLaVA (Zero-Shot)

This notebook demonstrates **zero-shot facial expression classification** using the [LLaVA v1.6 (Mistral-7B)](https://huggingface.co/llava-hf/llava-v1.6-mistral-7b-hf) multimodal model.

**Task:** Classify facial images into one of 7 emotion categories:
`angry` | `disgust` | `fear` | `happy` | `neutral` | `sad` | `surprised`

**Approach:** Zero-shot — no fine-tuning or additional training. The model is prompted directly with a classification instruction.

## 1. Imports & Model Loading

We load the **LLaVA-Next** processor and model from Hugging Face. The processor handles image and text preprocessing, while the model generates text outputs based on visual and textual inputs.

- **Precision:** `float16` to reduce GPU memory usage
- **Device:** Automatically selects GPU (CUDA) if available, otherwise CPU

In [ ]:
import torch
from tqdm.auto import tqdm
from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration

# --- Device Selection ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# --- Load Processor & Model ---
processor = LlavaNextProcessor.from_pretrained("llava-hf/llava-v1.6-mistral-7b-hf")

model = LlavaNextForConditionalGeneration.from_pretrained(
    "llava-hf/llava-v1.6-mistral-7b-hf",
    torch_dtype=torch.float16,       # Use 16-bit precision to reduce memory usage
    low_cpu_mem_usage=True,           # Minimize CPU RAM consumption during loading
    device_map="auto"                 # Automatically distribute model across available devices
)

## 2. Zero-Shot Classification

For each uploaded image, we:
1. Convert it to RGB format
2. Construct a prompt asking the model to classify the facial expression
3. Generate a prediction using the LLaVA model
4. Collect all results and export to an Excel file

**Prompt template:**
```
Classify this image into one of the given classes and describe it in one word:
angry, disgust, fear, happy, neutral, sad, surprised
```

In [ ]:
import os
import glob
from PIL import Image
import pandas as pd

# --- Configuration ---
IMAGE_DIR = "../data/test_images/"    # Path to the folder containing input images
OUTPUT_PATH = "../results/classification_results.xlsx"

# Supported image extensions
IMAGE_EXTENSIONS = ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp"]

# --- Collect image paths ---
image_paths = []
for ext in IMAGE_EXTENSIONS:
    image_paths.extend(glob.glob(os.path.join(IMAGE_DIR, ext)))

print(f"Found {len(image_paths)} images in '{IMAGE_DIR}'")

# --- Define prompt ---
prompt = "Classify this image into one of the given classes and describe it in one word: angry, disgust, fear, happy, neutral, sad, surprised"
formatted_prompt = f"[INST] <image>\n{prompt} [/INST]"

# --- Run classification ---
results = []

for image_path in tqdm(image_paths, desc="Classifying"):
    try:
        image_name = os.path.basename(image_path)

        # Open image and convert to RGB
        image = Image.open(image_path).convert("RGB")

        # Preprocess: convert image + text into model-ready tensors
        inputs = processor(images=image, text=formatted_prompt, return_tensors="pt").to(device)

        # Generate prediction
        output = model.generate(**inputs, max_new_tokens=100)
        output_text = processor.decode(output[0], skip_special_tokens=True)

        print(f"{image_name} -> {output_text}")

        results.append({
            "Image Name": image_name,
            "Classification": output_text
        })

    except Exception as e:
        print(f"Skipping {image_name}, error: {e}")
        continue

# --- Export results ---
df = pd.DataFrame(results)

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df.to_excel(OUTPUT_PATH, index=False)

print(f"\nAll images processed. Results saved to '{OUTPUT_PATH}'")

## 3. Results

The classification results are saved to `results/classification_results.xlsx`.

Each row contains:
- **Image Name** — the filename of the input image
- **Classification** — the model's predicted emotion label

### Limitations
- Zero-shot performance depends heavily on the prompt design
- The model was not fine-tuned on facial expression datasets
- Results may vary with image quality, lighting, and face orientation